# 01 · Preprocessing
**Input :** `data/raw/HCC1/` and `data/raw/HCC2/` (MTX triplets)  
**Output:** `data/processed/adata_processed.h5ad`

Covers: data loading, QC metrics, cell filtering, normalisation, log-transform,
and highly variable gene selection.

**Run order:** **01** → 02 → 03 → 04 → 05


In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "paths.py").exists():
            return p
    raise FileNotFoundError("paths.py not found — run: python scripts/data_download.py")

REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(REPO_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "scripts"))

from paths import REPO_ROOT, RAW_DIR, PROC_DIR, FIGURES_DIR, TABLES_DIR, MODELS_DIR
print(f"Repo root : {REPO_ROOT}")
print(f"Raw data  : {RAW_DIR}")

In [ ]:
import scanpy as sc
from scrna_functions import (load_samples, qc_metrics, filter_cells,
                              normalize, select_hvg, save_adata)
print(f"Scanpy: {sc.__version__}")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
MIN_GENES   = 200
MAX_GENES   = 2500
MAX_MT_PCT  = 5       # % mitochondrial reads
N_TOP_GENES = 2000    # highly variable genes to select

## 1 · Load data

In [ ]:
adata = load_samples(RAW_DIR)
adata

## 2 · QC metrics

In [ ]:
adata = qc_metrics(adata)
sc.pl.violin(adata, ["n_genes_by_counts","total_counts","pct_counts_mt"],
             groupby="sample", multi_panel=True)
sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="sample")

## 3 · Filter cells

In [ ]:
adata = filter_cells(adata, MIN_GENES, MAX_GENES, MAX_MT_PCT)

## 4 · Normalise & log-transform

In [ ]:
adata = normalize(adata)

## 5 · Highly variable genes

In [ ]:
adata = select_hvg(adata, n_top_genes=N_TOP_GENES, batch_key='sample')

## 6 · Save

In [ ]:
save_adata(adata, PROC_DIR / 'adata_processed.h5ad')